In [1]:
import csv
import pyodbc as odbc


In [2]:
# 1. PASSO: GERAR O FICHEIRO CSV COM AS 100 AMOSTRAS NOVAS

def gerar_csv_amostras():
    with open('amostras_vda.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        # Cabeçalho exigido pelo enunciado para abranger factos e dimensões
        writer.writerow(['variantID', 'chromosome', 'sourceID', 'sourceName', 
                         'diseaseID', 'diseaseName', 'typeID', 'typeName', 'score'])
        
        # Gerar exatamente 100 linhas com identificadores novos
        for i in range(1, 101):
            writer.writerow([
                f'rs_mock_vda_{i}',         # Nova Variant
                f'{ (i % 22) + 1 }',        # Cromossoma (1 a 23)
                f'SRC_VDA_{i % 5}',         # Nova Fonte
                f'Source_VDA_Channel_{i % 5}',
                f'C_MOCK_VDA_{i}',          # Nova Doença
                f'Nova Patologia VDA Tipo {i}',
                f'T_VDA_{i % 3}',           # Novo Tipo de Doença
                f'Classe_VDA_Categoria_{i % 3}',
                round(0.15 + (i * 0.006), 3) # Score numérico
            ])
    print("Ficheiro 'amostras_vda.csv' com 100 amostras gerado com sucesso.")


# 2. PASSO: CRIAR TABELAS E CARREGAR DADOS NO SERVIDOR DO DETI

def carregar_amostras_sql_server():
    server = 'deti-sql-aulas.ua.pt'
    database = 'team_09'
    username = 'team_09'
    password = 'spring'
    
    connString = f'DRIVER={{SQL Server}};SERVER={server};DATABASE={database};UID={username};PWD={password}'
    
    try:
        print("A estabelecer ligação ao SQL Server da equipa...")
        conn = odbc.connect(connString)
        cursor = conn.cursor()
        
        # Garante a criação física das tabelas conceituais pedidas no enunciado (padrão T-SQL)
        print("A verificar/criar as tabelas de dimensões e factos no servidor...")
        cursor.execute("""
            IF OBJECT_ID('dbo.DIM_Type', 'U') IS NULL
            CREATE TABLE dbo.DIM_Type (typeID VARCHAR(50) PRIMARY KEY, typeName VARCHAR(100));
        """)
        cursor.execute("""
            IF OBJECT_ID('dbo.DIM_Disease', 'U') IS NULL
            CREATE TABLE dbo.DIM_Disease (diseaseID VARCHAR(50) PRIMARY KEY, diseaseName VARCHAR(250), typeID VARCHAR(50));
        """)
        cursor.execute("""
            IF OBJECT_ID('dbo.DIM_Source', 'U') IS NULL
            CREATE TABLE dbo.DIM_Source (sourceID VARCHAR(50) PRIMARY KEY, sourceName VARCHAR(100));
        """)
        cursor.execute("""
            IF OBJECT_ID('dbo.DIM_Variant', 'U') IS NULL
            CREATE TABLE dbo.DIM_Variant (variantID VARCHAR(50) PRIMARY KEY, chromosome VARCHAR(50));
        """)
        cursor.execute("""
            IF OBJECT_ID('dbo.FACT_VDA', 'U') IS NULL
            CREATE TABLE dbo.FACT_VDA (variantID VARCHAR(50), sourceID VARCHAR(50), diseaseID VARCHAR(50), score REAL);
        """)
        
        # Ler o ficheiro CSV e processar linha a linha
        print("A ler o ficheiro CSV e a injetar dados de forma consistente...")
        with open('amostras_vda.csv', 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            next(reader)  # Saltar a linha de cabeçalho
            
            for linha in reader:
                var_id, chrom, src_id, src_name, dis_id, dis_name, typ_id, typ_name, score = linha
                
                # Inserção segura nas dimensões (Usa IF NOT EXISTS para evitar duplicações de chaves primárias)
                cursor.execute("IF NOT EXISTS (SELECT 1 FROM dbo.DIM_Type WHERE typeID = ?) INSERT INTO dbo.DIM_Type VALUES (?, ?)", (typ_id, typ_id, typ_name))
                cursor.execute("IF NOT EXISTS (SELECT 1 FROM dbo.DIM_Disease WHERE diseaseID = ?) INSERT INTO dbo.DIM_Disease VALUES (?, ?, ?)", (dis_id, dis_id, dis_name, typ_id))
                cursor.execute("IF NOT EXISTS (SELECT 1 FROM dbo.DIM_Source WHERE sourceID = ?) INSERT INTO dbo.DIM_Source VALUES (?, ?)", (src_id, src_id, src_name))
                cursor.execute("IF NOT EXISTS (SELECT 1 FROM dbo.DIM_Variant WHERE variantID = ?) INSERT INTO dbo.DIM_Variant VALUES (?, ?)", (var_id, var_id, chrom))
                
                # Inserção na Tabela de Factos FACT_VDA
                cursor.execute("INSERT INTO dbo.FACT_VDA (variantID, sourceID, diseaseID, score) VALUES (?, ?, ?, ?)", (var_id, src_id, dis_id, float(score)))
                
        conn.commit()
        print("-> Sucesso absoluto! As 100 amostras foram totalmente integradas nas tabelas do SQL Server.")
        
    except odbc.Error as e:
        if conn:
            conn.rollback()
        print("\n[ERRO DETETADO] Operação cancelada na totalidade:")
        print(e.args[1])
    finally:
        conn.close()

if __name__ == "__main__":
    gerar_csv_amostras()
    carregar_amostras_sql_server()

Ficheiro 'amostras_vda.csv' com 100 amostras gerado com sucesso.
A estabelecer ligação ao SQL Server da equipa...
A verificar/criar as tabelas de dimensões e factos no servidor...
A ler o ficheiro CSV e a injetar dados de forma consistente...
-> Sucesso absoluto! As 100 amostras foram totalmente integradas nas tabelas do SQL Server.
